# mRmR Feature Selection — Example Notebook

Demonstrates **P1 `mRmRSelector`** on the SCANB gene-expression dataset.

| Task | Target | Type | Relevance | Mode |
|------|--------|------|-----------|------|
| **A** | `is_lumA` | classification | `pearson` | precomputed matrix |
| **B** | `Lympho` | regression | `pearson` | precomputed matrix |
| **C** | `is_lumA` | classification | `mutual_information`, `random_forest`, `f_statistic` | precomputed relevance files |
| **D** | `Lympho` | regression | `mutual_information`, `random_forest`, `f_statistic` | precomputed relevance files |
| **E** | both | both | all methods | lazy hot-cache, 10 features, timing comparison |

**P1 features demonstrated:**
- `eval_every_k` — evaluate every 5 steps (Tasks A–D use 50 features)
- `'step'` key in `performance_history`
- `LR_random_baseline / LR_regression_baseline` with `return_summary=True`
- `selector.plot_vs_random_baseline(result, baseline_summary)`
- `FeatureRelevanceScores` — precomputed non-correlation relevance
- Lazy hot-cache timing comparison (Task E)

## 0 — Imports & paths

In [ ]:
import sys, os

# Make sure the package root is on the path when running from feature_selection/src/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from feature_selection.src.preprocessing import train_test_val_split
from feature_selection.src.filter.mRmR import mRmRSelector, SelectionResult
from feature_selection.src.LR_random_baseline import plot_performance_with_stats as clf_baseline_plot
from feature_selection.src.LR_regression_baseline import plot_performance_with_stats as reg_baseline_plot

# ── Paths (relative to notebook location: feature_selection/src/) ──────────
DATA_PATH  = '../data/SCANB.csv'
LABEL_PATH = '../data/sampleinfo_SCANB_t.csv'
CORR_PATH  = '../data/correlation_matrix.csv'  # 9 266 features + is_lumA + Lympho

# Precomputed relevance score files (Tasks C / D) — created on first run
REL_MI_CLF_PATH  = '../data/rel_mi_clf.csv'
REL_RF_CLF_PATH  = '../data/rel_rf_clf.csv'
REL_FS_CLF_PATH  = '../data/rel_fs_clf.csv'
REL_MI_REG_PATH  = '../data/rel_mi_reg.csv'
REL_RF_REG_PATH  = '../data/rel_rf_reg.csv'
REL_FS_REG_PATH  = '../data/rel_fs_reg.csv'

RANDOM_SEED  = 2
N_FEATURES   = 50    # Tasks A–D
EVAL_EVERY_K = 5     # evaluate every 5 steps
N_RUNS       = 10    # baseline runs

## 1 — Load data & build train / validation splits

In [ ]:
gene_expression_df = pd.read_csv(DATA_PATH)
sampleinfo_df      = pd.read_csv(LABEL_PATH)

# Binary LumA label
sampleinfo_df['is_lumA'] = (sampleinfo_df['PAM50'] == 'LumA').astype(int)

# 70 % train / 10 % val / 20 % test
train_si, val_si, _ = train_test_val_split(
    sampleinfo_df, random_seed=RANDOM_SEED, train_pcnt=0.7, val_pct=0.1
)

train_labels_df = train_si[['samplename', 'is_lumA']].reset_index(drop=True)
val_labels_df   = val_si[['samplename', 'is_lumA']].reset_index(drop=True)

print(f'Train samples : {len(train_labels_df)}')
print(f'Val   samples : {len(val_labels_df)}')
print(f"LumA prevalence (train): {train_labels_df['is_lumA'].mean():.2%}")

# Transpose gene-expression (genes × samples) → (samples × genes)
ge_proc = gene_expression_df.copy().fillna(0).set_index('Unnamed: 0').T
ge_proc.index.name = 'samplename'

X_train = ge_proc.loc[train_labels_df['samplename']]
X_val   = ge_proc.loc[val_labels_df['samplename']]

print(f'\nX_train shape : {X_train.shape}')
print(f'X_val   shape : {X_val.shape}')

## 2 — Build target series

Both targets are already stored in `correlation_matrix.csv` under their real column names.

In [ ]:
# Classification target: is_lumA
y_lumA = (
    train_labels_df.set_index('samplename')['is_lumA']
    .reindex(X_train.index)
)  # Series.name == 'is_lumA'

y_lumA_val = (
    val_labels_df.set_index('samplename')['is_lumA']
    .reindex(X_val.index)
)

# Regression target: Lympho
y_lympho = (
    sampleinfo_df.set_index('samplename')['Lympho']
    .reindex(X_train.index)
)  # Series.name == 'Lympho'

y_lympho_val = (
    sampleinfo_df.set_index('samplename')['Lympho']
    .reindex(X_val.index)
)

print(f'y_lumA   name={y_lumA.name!r}    dtype={y_lumA.dtype}   nunique={y_lumA.nunique()}')
print(f'y_lympho name={y_lympho.name!r}  dtype={y_lympho.dtype}   nunique={y_lympho.nunique()}')

---

## Task A — Classification (Pearson, precomputed matrix, 50 features, eval every 5)

### A.1 — Run mRmR selector

In [ ]:
selector_clf = mRmRSelector(
    X_train=X_train,
    y_train=y_lumA,
    relevance_method='pearson',
    mrmr_score_method='difference',
    correlation_filepath=CORR_PATH,
    gene_expression_df=gene_expression_df,
    train_labels_df=train_labels_df,
    val_labels_df=val_labels_df,
    random_seed=RANDOM_SEED,
)

result_clf: SelectionResult = selector_clf.forward_selection(
    n_features_to_select=N_FEATURES,
    eval_every_k=EVAL_EVERY_K,
)

print()
print(result_clf)

### A.2 — Selected features

In [ ]:
print('Classification — selected features (in order):')
for i, f in enumerate(result_clf.selected_features, 1):
    print(f'  {i:2d}. {f}')

In [ ]:
# Inspect first history entry — includes 'step' key (P1 G1)
result_clf.performance_history[0]

### A.3 — LR random baseline

In [ ]:
# Checkpoints align with evaluated steps (every 5th)
checkpoints_clf = [entry['step'] for entry in result_clf.performance_history]

baseline_summary_clf = clf_baseline_plot(
    gene_expression_df=gene_expression_df,
    train_labels_df=train_labels_df,
    val_labels_df=val_labels_df,
    N_values=checkpoints_clf,
    random_seed=RANDOM_SEED,
    num_runs=N_RUNS,
    return_summary=True,
)

### A.4 — Comparison plot

In [ ]:
selector_clf.plot_vs_random_baseline(
    result=result_clf,
    baseline_summary=baseline_summary_clf,
    title_suffix=' — SCANB (Pearson, precomputed)',
)

---

## Task B — Regression (Pearson, precomputed matrix, 50 features, eval every 5)

### B.1 — Run mRmR selector

In [ ]:
selector_reg = mRmRSelector(
    X_train=X_train,
    y_train=y_lympho,
    relevance_method='pearson',
    mrmr_score_method='difference',
    correlation_filepath=CORR_PATH,
    random_seed=RANDOM_SEED,
    X_val=X_val,
    y_val=y_lympho_val,
)

result_reg: SelectionResult = selector_reg.forward_selection(
    n_features_to_select=N_FEATURES,
    eval_every_k=EVAL_EVERY_K,
)

print()
print(result_reg)

### B.2 — Selected features & relevance bar chart

In [ ]:
print('Regression — selected features (in order):')
for i, f in enumerate(result_reg.selected_features, 1):
    print(f'  {i:2d}. {f}')

target_col = selector_reg._target_col
rel_scores = {
    f: selector_reg._score_matrix.loc[f, target_col]
    for f in result_reg.selected_features
}
rel_df = (
    pd.Series(rel_scores, name='|Pearson| with Lympho')
    .reset_index().rename(columns={'index': 'feature'})
    .sort_values('|Pearson| with Lympho', ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(rel_df['feature'][::-1], rel_df['|Pearson| with Lympho'][::-1],
        color='teal', alpha=0.8)
ax.set_xlabel('|Pearson correlation with Lympho|')
ax.set_title(f'Top {N_FEATURES} mRmR features — Regression task (Lympho)', fontsize=12)
ax.grid(True, axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

display(rel_df)

### B.3 — LinReg random baseline

In [ ]:
checkpoints_reg = [entry['step'] for entry in result_reg.performance_history]

baseline_summary_reg = reg_baseline_plot(
    X_train=X_train,
    y_train=y_lympho,
    X_val=X_val,
    y_val=y_lympho_val,
    N_values=checkpoints_reg,
    random_seed=RANDOM_SEED,
    num_runs=N_RUNS,
    return_summary=True,
)

### B.4 — Comparison plot

In [ ]:
selector_reg.plot_vs_random_baseline(
    result=result_reg,
    baseline_summary=baseline_summary_reg,
    title_suffix=' — SCANB (Pearson, precomputed)',
)

---

## Task C — Classification: non-correlation relevance methods (precomputed files)

Runs `mutual_information`, `random_forest`, and `f_statistic` for the `is_lumA` target.  
Each selector uses a precomputed relevance scores file (created on first run, reloaded thereafter).  
Redundancy uses the same precomputed Pearson matrix (`CORR_PATH`).  
50 features selected, evaluation every 5 steps.  Performance histories are compared across methods.

### C.1 — Run all three non-correlation selectors

In [ ]:
non_corr_methods_clf = [
    ('mutual_information', REL_MI_CLF_PATH),
    ('random_forest',      REL_RF_CLF_PATH),
    ('f_statistic',        REL_FS_CLF_PATH),
]

results_C = {}   # method → SelectionResult

for method, rel_path in non_corr_methods_clf:
    print(f'\n=== Classification | relevance_method={method!r} ===')
    sel = mRmRSelector(
        X_train=X_train,
        y_train=y_lumA,
        relevance_method=method,
        redundancy_method='pearson',        # pairwise correlation for redundancy
        mrmr_score_method='difference',
        correlation_filepath=CORR_PATH,     # reuse Pearson matrix for redundancy
        relevance_scores_filepath=rel_path, # precomputed file (Mode A)
        gene_expression_df=gene_expression_df,
        train_labels_df=train_labels_df,
        val_labels_df=val_labels_df,
        random_seed=RANDOM_SEED,
    )
    result = sel.forward_selection(
        n_features_to_select=N_FEATURES,
        eval_every_k=EVAL_EVERY_K,
    )
    results_C[method] = result
    print(result)

### C.2 — Compare performance history across methods

In [ ]:
# Also include the Pearson result from Task A for reference
all_clf_results = {
    'pearson (precomputed)': result_clf,
    **{m: results_C[m] for m, _ in non_corr_methods_clf},
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['steelblue', 'darkorange', 'seagreen', 'crimson']
metrics_to_plot = [
    ('accuracy', 'Accuracy'),
    ('macro_f1', 'Macro F1'),
]

for ax, (metric_key, metric_label) in zip(axes, metrics_to_plot):
    for (method_label, res), color in zip(all_clf_results.items(), colors):
        steps  = [e['step'] for e in res.performance_history]
        if metric_key == 'accuracy':
            values = [e.get('accuracy', float('nan')) for e in res.performance_history]
        else:
            values = [
                e.get('macro avg', {}).get('f1-score', float('nan'))
                if isinstance(e.get('macro avg'), dict) else float('nan')
                for e in res.performance_history
            ]
        ax.plot(steps, values, marker='o', linewidth=2, color=color, label=method_label)
    ax.set_title(f'Classification — {metric_label}', fontsize=13)
    ax.set_xlabel('Number of features selected')
    ax.set_ylabel(metric_label)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)

fig.suptitle(
    'Classification (is_lumA): Relevance Method Comparison — SCANB',
    fontsize=13, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

---

## Task D — Regression: non-correlation relevance methods (precomputed files)

Same structure as Task C but for the `Lympho` regression target.  
Performance histories (R², MSE, MAE) are compared across methods.

### D.1 — Run all three non-correlation selectors

In [ ]:
non_corr_methods_reg = [
    ('mutual_information', REL_MI_REG_PATH),
    ('random_forest',      REL_RF_REG_PATH),
    ('f_statistic',        REL_FS_REG_PATH),
]

results_D = {}   # method → SelectionResult

for method, rel_path in non_corr_methods_reg:
    print(f'\n=== Regression | relevance_method={method!r} ===')
    sel = mRmRSelector(
        X_train=X_train,
        y_train=y_lympho,
        relevance_method=method,
        redundancy_method='pearson',
        mrmr_score_method='difference',
        correlation_filepath=CORR_PATH,
        relevance_scores_filepath=rel_path,
        random_seed=RANDOM_SEED,
        X_val=X_val,
        y_val=y_lympho_val,
    )
    result = sel.forward_selection(
        n_features_to_select=N_FEATURES,
        eval_every_k=EVAL_EVERY_K,
    )
    results_D[method] = result
    print(result)

### D.2 — Compare performance history across methods

In [ ]:
all_reg_results = {
    'pearson (precomputed)': result_reg,
    **{m: results_D[m] for m, _ in non_corr_methods_reg},
}

reg_metrics = [
    ('r2',  'R²'),
    ('mse', 'MSE'),
    ('mae', 'MAE'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (metric_key, metric_label) in zip(axes, reg_metrics):
    for (method_label, res), color in zip(all_reg_results.items(), colors):
        steps  = [e['step'] for e in res.performance_history]
        values = [e.get(metric_key, float('nan')) for e in res.performance_history]
        ax.plot(steps, values, marker='o', linewidth=2, color=color, label=method_label)
    ax.set_title(f'Regression — {metric_label}', fontsize=13)
    ax.set_xlabel('Number of features selected')
    ax.set_ylabel(metric_label)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)

fig.suptitle(
    'Regression (Lympho): Relevance Method Comparison — SCANB',
    fontsize=13, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

---

## Task E — Lazy hot-cache timing comparison (10 features)

All relevance methods run **without** a precomputed file — scores are computed
lazily on first access and cached in memory (`_relevance_cache` / `_score_matrix`).

- Pearson correlation: lazy hot-cache from the full `_score_matrix` (no precomputed file, standard Mode B)
- `mutual_information`, `random_forest`, `f_statistic`: lazy `_relevance_cache` per feature
- No evaluation (no LR/LinReg eval params) — `selection_time_seconds` measures pure mRmR scoring
- 10 features selected; average time per feature = `selection_time_seconds / 10`

Both classification and regression targets are timed and summarised together.

### E.1 — Run lazy-cache selectors (classification)

In [ ]:
N_LAZY = 10   # number of features for timing experiment

# Methods to bench for classification (pearson only from correlation family)
lazy_methods_clf = [
    ('pearson',            None),   # correlation — lazy Mode B, no relevance_scores_filepath
    ('mutual_information', None),
    ('random_forest',      None),
    ('f_statistic',        None),
]

timing_records_clf = []

for method, _ in lazy_methods_clf:
    kwargs = dict(
        X_train=X_train,
        y_train=y_lumA,
        relevance_method=method,
        mrmr_score_method='difference',
        random_seed=RANDOM_SEED,
        # No correlation_filepath → lazy Mode B for correlation method too
        # No relevance_scores_filepath → lazy hot-cache for non-corr methods
    )
    if method in ('mutual_information', 'random_forest', 'f_statistic'):
        kwargs['redundancy_method'] = 'pearson'

    sel = mRmRSelector(**kwargs)
    res = sel.forward_selection(n_features_to_select=N_LAZY)

    avg_time = res.selection_time_seconds / N_LAZY
    timing_records_clf.append({
        'method': method,
        'task': 'classification',
        'total_selection_time_s': round(res.selection_time_seconds, 4),
        'avg_time_per_feature_s': round(avg_time, 4),
    })
    print(f'{method:25s}  total={res.selection_time_seconds:.3f}s  avg/feature={avg_time:.4f}s')

### E.2 — Run lazy-cache selectors (regression)

In [ ]:
lazy_methods_reg = [
    ('pearson',            None),
    ('mutual_information', None),
    ('random_forest',      None),
    ('f_statistic',        None),
]

timing_records_reg = []

for method, _ in lazy_methods_reg:
    kwargs = dict(
        X_train=X_train,
        y_train=y_lympho,
        relevance_method=method,
        mrmr_score_method='difference',
        random_seed=RANDOM_SEED,
    )
    if method in ('mutual_information', 'random_forest', 'f_statistic'):
        kwargs['redundancy_method'] = 'pearson'

    sel = mRmRSelector(**kwargs)
    res = sel.forward_selection(n_features_to_select=N_LAZY)

    avg_time = res.selection_time_seconds / N_LAZY
    timing_records_reg.append({
        'method': method,
        'task': 'regression',
        'total_selection_time_s': round(res.selection_time_seconds, 4),
        'avg_time_per_feature_s': round(avg_time, 4),
    })
    print(f'{method:25s}  total={res.selection_time_seconds:.3f}s  avg/feature={avg_time:.4f}s')

### E.3 — Timing summary table

In [ ]:
timing_df = pd.DataFrame(timing_records_clf + timing_records_reg)
print('Lazy hot-cache selection timing — 10 features:')
display(timing_df.set_index(['task', 'method']))

### E.4 — Timing bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (task, records) in zip(axes, [
    ('classification', timing_records_clf),
    ('regression',     timing_records_reg),
]):
    methods = [r['method'] for r in records]
    avgs    = [r['avg_time_per_feature_s'] for r in records]
    bar_colors = ['steelblue', 'darkorange', 'seagreen', 'crimson']
    bars = ax.bar(methods, avgs, color=bar_colors[:len(methods)], alpha=0.85)
    ax.bar_label(bars, fmt='{:.4f}s', padding=3, fontsize=9)
    ax.set_title(f'{task.capitalize()} — avg selection time per feature\n(lazy hot-cache, 10 features)', fontsize=12)
    ax.set_xlabel('Relevance method')
    ax.set_ylabel('Avg time per feature (s)')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(True, axis='y', alpha=0.4)

fig.suptitle(
    'Task E — Lazy Hot-Cache Timing Comparison (SCANB, 10 features)',
    fontsize=13, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

---

## Summary — Timing across all tasks

In [ ]:
print('=== Task A — Classification (Pearson, precomputed, 50 features) ===')
print(f'  Selection time  : {result_clf.selection_time_seconds:.3f}s')
if result_clf.evaluation_time_seconds is not None:
    print(f'  Evaluation time : {result_clf.evaluation_time_seconds:.3f}s  '
          f'({len(result_clf.performance_history)} LR fits)')
print(f'  Stopping reason : {result_clf.stopping_reason}')

print()
print('=== Task B — Regression (Pearson, precomputed, 50 features) ===')
print(f'  Selection time  : {result_reg.selection_time_seconds:.3f}s')
if result_reg.evaluation_time_seconds is not None:
    print(f'  Evaluation time : {result_reg.evaluation_time_seconds:.3f}s  '
          f'({len(result_reg.performance_history)} LinReg fits)')
print(f'  Stopping reason : {result_reg.stopping_reason}')

print()
print('=== Task C — Classification non-correlation methods (precomputed, 50 features) ===')
for method, res in results_C.items():
    print(f'  {method:25s}: sel={res.selection_time_seconds:.3f}s  '
          f'eval={res.evaluation_time_seconds:.3f}s  '
          f'evals={len(res.performance_history)}')

print()
print('=== Task D — Regression non-correlation methods (precomputed, 50 features) ===')
for method, res in results_D.items():
    print(f'  {method:25s}: sel={res.selection_time_seconds:.3f}s  '
          f'eval={res.evaluation_time_seconds:.3f}s  '
          f'evals={len(res.performance_history)}')

print()
print('=== Task E — Lazy hot-cache timing (10 features) ===')
display(timing_df.set_index(['task', 'method']))